In [6]:
from pathlib import Path
import random
from app.ingestion.pdf_loader import load_pdfs
from app.ingestion.text_cleaner import clean_text, clean_documents

from app.rag.chunker import SemantikChunker
from app.rag.embedding import EmbeddingManager
from app.rag.vector_store import VectorStore

In [2]:
BASE_DIR = Path.cwd()
RAW_DATA_DIR = BASE_DIR / "data" / "raw"

raw_data = load_pdfs(RAW_DATA_DIR)
cleaned_docs = clean_documents(raw_data)



In [3]:
chunker = SemantikChunker()
chunks = chunker.chunk_corpus(cleaned_docs)

print(f"Generated Chunk Count: {len(chunks)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7854.07it/s]


Generated Chunk Count: 2347


In [4]:
random_idx = random.randint(0, len(chunks) - 1)
sample_chunk = chunks[random_idx]

print(f"Index: {random_idx}")
print(f"Chunk ID: {sample_chunk.chunk_id}")
print(f"Source: {sample_chunk.source} (Page {sample_chunk.page})")
print(f"Text Character Count: {len(sample_chunk.text)}")
print("-" * 50)
print(sample_chunk.text)

Index: 1464
Chunk ID: NIST.AI.600-1_p59_c1
Source: NIST.AI.600-1.pdf (Page 59)
Text Character Count: 63
--------------------------------------------------
(2024) Short, Mid and Long-Term Impacts of AI in Cybersecurity.


In [5]:
processed_path = Path("data/processed/chunk.jsonl")
processed_path.parent.mkdir(parents=True, exist_ok=True)

with open(processed_path, "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(chunk.model_dump_json() + "\n")

print(f"The chunks have been saved:{processed_path} ({len(chunks)} Chunk)")

The chunks have been saved:data/processed/chunk.jsonl (2347 Chunk)


In [7]:
VECTORSTORE_DIR = BASE_DIR / "data" / "vectorstore"

embed_mgr = EmbeddingManager(model_name="all-MiniLM-L6-v2")
vector_store = VectorStore(
    persist_dir=VECTORSTORE_DIR,
    collection_name="domainforge_governance",
    embedding_manager=embed_mgr
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14464.09it/s]


In [8]:
vector_store.add_chunks(chunks)

print(f"Ready! Total number of records in the database: {vector_store.count()}")

[VECTOR STORE] 2347 chunks are being indexed to disk...


Batches: 100%|██████████| 4/4 [00:00<00:00, 18.37it/s]


[VECTOR STORE] Indexing completed and saved. Total: 2347
Ready! Total number of records in the database: 2347
